# Spread Analysis - South African ZAR Bonds (SAGB/SOAF)

In [1]:
# Set up your environment
from functools import partial

import ipydatagrid as ipd
import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import bql

In [2]:
# Connect to BQL
bq = bql.Service()

In [3]:
def get_universe(ticker):
    """
    Return a BQL universe for the given ticker.
    """
    universe = bq.univ.bondsuniv("active").filter(
        bq.data.ticker() == ticker.upper()
    )
    universe = universe.filter(
        bq.data.id_isin() != bql.NA
    )
    return universe

In [4]:
def get_maturity_buckets():
    """
    Return maturity buckets for bonds based on maturity in years.
    """
    bin_points = [1, 2, 3, 5, 7, 10, 20, 30]
    bin_names = [
        "<= 1 yrs",
        "1 to 2 yrs",
        "2 to 3 yrs",
        "3 to 5 yrs",
        "5 to 7 yrs",
        "7 to 10 yrs",
        "10 to 20 yrs",
        "20 to 30 yrs",
        "30+ yrs",
    ]
    bins = bq.data.maturity_years().bins(bin_points, bin_names)
    return bins

In [5]:
def get_bond_grid_data(ticker):
    """
    Creates a bond-level DataFrame with all XCCY spreads for USD, EUR, GBP.
    """
    universe = get_universe(ticker)

    data_items = {
        "ISIN": bq.data.id_isin(),
        "Name": bq.data.name(),
        "Ccy": bq.data.crncy(),
        "Cpn": bq.data.cpn(),
        "Issue Date": bq.data.issue_dt(),
        "Mty": bq.data.maturity(),
        "Mty Yrs": bq.data.maturity_years(),
        "Mty Bucket": get_maturity_buckets(),
        "Cpn Typ": bq.data.cpn_typ(),
        "Amt Out (Bn)": bq.data.amt_outstanding(currency="ZAR") / 1e9,
        "ASW": bq.data.spread(spread_type="ASW"),
        "Z Spd": bq.data.spread(spread_type="Z"),
        "XCCY ASW USD": bq.data.spread(spread_type="ASW").with_additional_parameters(cross_currency="USD"),
        "XCCY ASW EUR": bq.data.spread(spread_type="ASW").with_additional_parameters(cross_currency="EUR"),
        "XCCY ASW GBP": bq.data.spread(spread_type="ASW").with_additional_parameters(cross_currency="GBP"),
        "XCCY Z USD": bq.data.spread(spread_type="Z").with_additional_parameters(cross_currency="USD"),
        "XCCY Z EUR": bq.data.spread(spread_type="Z").with_additional_parameters(cross_currency="EUR"),
        "XCCY Z GBP": bq.data.spread(spread_type="Z").with_additional_parameters(cross_currency="GBP"),
        "LQA": bq.data.lqa_liquidity_score(),
    }

    request = bql.Request(universe, data_items)
    response = bq.execute(request)

    data = pd.concat(
        [data_item.df()[data_item.name] for data_item in response],
        axis=1,
    )
    
    # Replace Inf values with NaN to avoid JSON serialization errors
    data = data.replace([np.inf, -np.inf], np.nan)

    return data.sort_values(by="Mty")

In [6]:
def generate_spread_history(ticker, spread_type, bond_grid_data, start_date):
    """
    Fetch historical ASW or Z spreads.
    spread_type should be 'ASW' or 'Z Spd'
    """
    try:
        universe = get_universe(ticker)
        
        # Map display name to BQL spread type
        bql_spread_type = 'ASW' if spread_type == 'ASW' else 'Z'
        spread_field = bq.data.spread(spread_type=bql_spread_type)
        
        request = bql.Request(
            universe,
            {"Spread": spread_field},
            with_params={"dates": bq.func.range(start_date, bq.func.today())},
        )

        response = bq.execute(request)
        data = response[0].df()
        
        if data.empty:
            return pd.DataFrame()

        data = data.pivot_table(index="DATE", columns="ID", values="Spread")
        
        # Replace Inf with NaN
        data = data.replace([np.inf, -np.inf], np.nan)
        
        # Map to bond names
        name_map = dict(zip(bond_grid_data.index, bond_grid_data['Name']))
        data.columns = data.columns.map(lambda x: name_map.get(x, x))
        
        # Filter by issue date
        issue_date_map = dict(zip(bond_grid_data['Name'], bond_grid_data['Issue Date']))
        for col in data.columns:
            if col in issue_date_map and pd.notna(issue_date_map[col]):
                data.loc[data.index < issue_date_map[col], col] = np.nan

        return data
    except Exception as e:
        print(f"{spread_type} history fetch failed: {str(e)[:50]}...")
        return pd.DataFrame()

In [7]:
def generate_lqa_history(ticker, bond_grid_data, start_date):
    """
    Fetch historical LQA scores. Data only available from 2022-07-15.
    """
    try:
        universe = get_universe(ticker)
        
        # LQA data only available from 2022-07-15
        effective_start = max(start_date, '2022-07-15')
        
        request = bql.Request(
            universe,
            {"LQA": bq.data.lqa_liquidity_score()},
            with_params={"dates": bq.func.range(effective_start, bq.func.today())},
        )

        response = bq.execute(request)
        data = response[0].df()
        
        if data.empty:
            return pd.DataFrame()

        data = data.pivot_table(index="DATE", columns="ID", values="LQA")
        
        # Replace Inf with NaN
        data = data.replace([np.inf, -np.inf], np.nan)
        
        # Map to bond names
        name_map = dict(zip(bond_grid_data.index, bond_grid_data['Name']))
        data.columns = data.columns.map(lambda x: name_map.get(x, x))
        
        # Filter by issue date
        issue_date_map = dict(zip(bond_grid_data['Name'], bond_grid_data['Issue Date']))
        for col in data.columns:
            if col in issue_date_map and pd.notna(issue_date_map[col]):
                data.loc[data.index < issue_date_map[col], col] = np.nan

        return data
    except Exception as e:
        print(f"LQA history fetch failed: {str(e)[:50]}...")
        return pd.DataFrame()

In [8]:
def create_spread_heatmap_data(bond_grid_data, spread_history, spread_col, zscore_lookback):
    """
    Create heatmap data for spread differences (longer - shorter maturity).
    Returns DataFrames for current spread diff and z-score.
    """
    # Get bonds sorted by maturity
    df = bond_grid_data.copy()
    df = df.sort_values('Mty')
    bond_names = df['Name'].tolist()
    
    n = len(bond_names)
    diff_matrix = pd.DataFrame(index=bond_names, columns=bond_names, dtype=float)
    zscore_matrix = pd.DataFrame(index=bond_names, columns=bond_names, dtype=float)
    
    if spread_history.empty:
        return diff_matrix, zscore_matrix
    
    # Calculate differences (row - column, where row is longer maturity)
    for i, long_bond in enumerate(bond_names):
        for j, short_bond in enumerate(bond_names):
            if i <= j:  # Only fill upper triangle (longer - shorter)
                diff_matrix.loc[long_bond, short_bond] = np.nan
                zscore_matrix.loc[long_bond, short_bond] = np.nan
                continue
            
            if long_bond not in spread_history.columns or short_bond not in spread_history.columns:
                continue
            
            diff_series = spread_history[long_bond] - spread_history[short_bond]
            diff_series = diff_series.dropna()
            
            if len(diff_series) > 0:
                current_diff = diff_series.iloc[-1]
                diff_matrix.loc[long_bond, short_bond] = current_diff
                
                if len(diff_series) >= zscore_lookback:
                    rolling_mean = diff_series.rolling(zscore_lookback).mean().iloc[-1]
                    rolling_std = diff_series.rolling(zscore_lookback).std().iloc[-1]
                    if rolling_std > 0:
                        zscore_matrix.loc[long_bond, short_bond] = (current_diff - rolling_mean) / rolling_std
    
    return diff_matrix, zscore_matrix

In [9]:
def create_renderers():
    """
    Creates column renderers for the grid.
    """
    renderers = {
        'ASW': ipd.TextRenderer(format='.1f'),
        'Z Spd': ipd.TextRenderer(format='.1f'),
        'XCCY ASW USD': ipd.TextRenderer(format='.1f'),
        'XCCY ASW EUR': ipd.TextRenderer(format='.1f'),
        'XCCY ASW GBP': ipd.TextRenderer(format='.1f'),
        'XCCY Z USD': ipd.TextRenderer(format='.1f'),
        'XCCY Z EUR': ipd.TextRenderer(format='.1f'),
        'XCCY Z GBP': ipd.TextRenderer(format='.1f'),
        'Issue Date': ipd.TextRenderer(format='%Y-%m-%d', format_type='time'),
        'Mty': ipd.TextRenderer(format='%Y-%m-%d', format_type='time'),
        'Amt Out (Bn)': ipd.TextRenderer(format='.1f'),
        'Cpn': ipd.TextRenderer(format='.3f'),
        'LQA': ipd.TextRenderer(format='.0f'),
        'Mty Yrs': ipd.TextRenderer(format='.2f'),
        'Chart': ipd.TextRenderer(
            text_value=ipd.VegaExpr("cell.value === 0 ? '\u2610' : '\u2611'"),
            horizontal_alignment='center',
            font='16px Arial'
        ),
        'Anchor': ipd.TextRenderer(
            text_value=ipd.VegaExpr("cell.value === 0 ? ' ' : '\u2611'"),
            horizontal_alignment='center',
            font='16px Arial'
        ),
        'Comparable': ipd.TextRenderer(
            text_value=ipd.VegaExpr("cell.value === 0 ? ' ' : '\u2611'"),
            horizontal_alignment='center',
            font='16px Arial'
        )
    }
    return renderers

In [10]:
def create_grid(df):
    """
    Builds the main bond grid with all XCCY spreads.
    """
    df = df.sort_values(by='Mty', ascending=True)
    df = df.set_index('Name')
    df = df.drop(columns=['ISIN'])

    # Add checkbox columns
    df.insert(0, 'Chart', 0)
    df.insert(1, 'Anchor', 0)
    df.insert(2, 'Comparable', 0)

    renderers = create_renderers()

    grid = ipd.DataGrid(
        dataframe=df,
        renderers=renderers,
        header_renderer=ipd.TextRenderer(
            text_wrap=True,
            vertical_alignment="top",
            background_color="rgb(66,66,66)",
            horizontal_alignment="center"
        ),
        base_column_header_size=45,
        base_row_header_size=140,
        column_widths={
            'Chart': 45,
            'Anchor': 50,
            'Comparable': 70,
            'Ccy': 35,
            'Cpn': 45,
            'Issue Date': 80,
            'Mty': 80,
            'Mty Yrs': 50,
            'Cpn Typ': 50,
            'Amt Out (Bn)': 55,
            'Mty Bucket': 80,
            'LQA': 40,
            'ASW': 45,
            'Z Spd': 45,
            'XCCY ASW USD': 65,
            'XCCY ASW EUR': 65,
            'XCCY ASW GBP': 65,
            'XCCY Z USD': 60,
            'XCCY Z EUR': 60,
            'XCCY Z GBP': 60,
        },
        layout={
            'height': '500px',
            'width': '100%',
        },
        selection_mode='cell',
    )

    return grid

In [11]:
# Input widgets
banner = widgets.HTML("<h2>Spread Analysis - SA ZAR Bonds</h2>")

ticker = widgets.Text(
    value="SAGB",
    description="Ticker",
    layout={'width': '150px'},
    style={'description_width': '50px'}
)

spread_chart_type = widgets.Dropdown(
    description="Chart",
    options=["ASW", "Z Spd"],
    value="ASW",
    layout={'width': '150px'},
    style={'description_width': '50px'}
)

zscore_lookback = widgets.IntText(
    description='Z Lookback',
    value=90,
    layout={'width': '150px'},
    style={'description_width': '70px'}
)

start_date = widgets.Text(
    value="2022-01-01",
    description="Start",
    layout={'width': '150px'},
    style={'description_width': '40px'}
)

go_button = widgets.Button(
    description='Go', 
    layout={'width': '50px'},
    button_style='success'
)

spinner = widgets.HTML(
    '''<i class="fa fa-spinner fa-spin" style="font-size: 18px"></i>''',
    layout={'visibility': 'hidden', 'margin': '5px 0 0 10px'}
)

exception = widgets.HTML()

control_box = widgets.HBox(
    [banner, ticker, spread_chart_type, zscore_lookback, start_date, go_button, spinner, exception],
    layout={'align_items': 'center', 'flex_wrap': 'wrap'}
)

In [12]:
def get_next_color(figure):
    """Get next unused color."""
    used = [t.line.color.upper() for t in figure.data if hasattr(t, 'line') and t.line.color]
    for c in px.colors.qualitative.Plotly:
        if c.upper() not in used:
            return c
    return px.colors.qualitative.Plotly[0]


def create_legend_str(row):
    """Create legend string."""
    return row['Ccy'].item() + ' - ' + row.index.item()

In [13]:
def update_spread_trace(bond_name, legend_str, spread_history, fig, show):
    """Add or remove spread trace."""
    if spread_history.empty or bond_name not in spread_history.columns:
        return
    
    if not show:
        series = spread_history[bond_name].dropna()
        if len(series) > 0:
            fig.add_trace(
                go.Scatter(
                    x=series.index, y=series.values,
                    name=legend_str,
                    meta={'bond_name': bond_name},
                    showlegend=True,
                    line=dict(width=1, color=get_next_color(fig)),
                ),
                row=1, col=1
            )
    else:
        fig.data = tuple(t for t in fig.data if t.name != legend_str)


def update_lqa_trace(bond_name, legend_str, lqa_history, fig, show):
    """Add or remove LQA trace."""
    if lqa_history.empty or bond_name not in lqa_history.columns:
        return
    
    if not show:
        series = lqa_history[bond_name].dropna()
        if len(series) > 0:
            fig.add_trace(
                go.Scatter(
                    x=series.index, y=series.values,
                    name=legend_str,
                    meta={'bond_name': bond_name},
                    showlegend=True,
                    line=dict(width=1, color=get_next_color(fig)),
                )
            )
    else:
        fig.data = tuple(t for t in fig.data if t.name != legend_str)

In [14]:
def update_checkbox_anchor_comp(grid, clicked_security, col_name):
    """
    Single-selection for Anchor/Comparable. Mutually exclusive.
    """
    df = grid.data
    
    # Clear previous in same column
    prev = df[df[col_name] == 1]
    if len(prev) == 1:
        grid.set_cell_value(col_name, prev.index[0], 0)
    
    # Clear from other column if selected there
    other_col = 'Comparable' if col_name == 'Anchor' else 'Anchor'
    if df.loc[clicked_security, other_col] == 1:
        grid.set_cell_value(other_col, clicked_security, 0)
    
    grid.set_cell_value(col_name, clicked_security, 1)


def update_checkbox_chart(grid, clicked_bond_name, value):
    """Toggle Chart checkbox."""
    grid.set_cell_value('Chart', clicked_bond_name, abs(value - 1))

In [15]:
def update_anchor_comparable(grid, spread_history, spread_fig, lookback):
    """
    Update difference and z-score charts.
    """
    if spread_history.empty:
        return
    
    df = grid.data
    anchor = df[df['Anchor'] == 1]
    comparable = df[df['Comparable'] == 1]
    
    if len(anchor) != 1 or len(comparable) != 1:
        return
    
    anchor_name = anchor.index.item()
    comp_name = comparable.index.item()
    
    if anchor_name not in spread_history.columns or comp_name not in spread_history.columns:
        return
    
    diff = (spread_history[anchor_name] - spread_history[comp_name]).dropna()
    
    if len(diff) < lookback:
        return
    
    rolling = diff.rolling(lookback, min_periods=lookback)
    zscore = ((diff - rolling.mean()) / rolling.std()).dropna()
    
    anchor_legend = create_legend_str(anchor)
    comp_legend = create_legend_str(comparable)
    
    with spread_fig.batch_update():
        # Update difference trace (row 2)
        spread_fig.update_traces(row=2, col=1, patch=dict(x=diff.index, y=diff.values))
        if len(diff) > 0:
            maxx = float(np.nanmax(np.abs(diff.values))) * 1.1
            spread_fig.update_yaxes(range=[-maxx if maxx > 0 else -1, maxx if maxx > 0 else 1], row=2, col=1)
        
        # Update z-score trace (row 3)
        spread_fig.update_traces(row=3, col=1, patch=dict(x=zscore.index, y=zscore.values))
        if len(zscore) > 0:
            maxx = float(np.nanmax(np.abs(zscore.values))) * 1.1
            spread_fig.update_yaxes(range=[-maxx if maxx > 0 else -1, maxx if maxx > 0 else 1], row=3, col=1)
        
        spread_fig.layout.annotations[1].text = f"{anchor_legend} vs {comp_legend}"

In [16]:
def on_grid_click(grid, spread_history, lqa_history, spread_fig, lqa_fig, lookback, event):
    """
    Handle grid click events.
    """
    col_name = event['column']
    value = event['cell_value']
    clicked_row = grid.data.iloc[[event['primary_key_row']]]
    clicked_bond = clicked_row.index.item()

    if col_name == 'Chart':
        legend_str = create_legend_str(clicked_row)
        update_spread_trace(clicked_bond, legend_str, spread_history, spread_fig, show=value)
        update_lqa_trace(clicked_bond, legend_str, lqa_history, lqa_fig, show=value)
        update_checkbox_chart(grid, clicked_bond, value)

    elif col_name in ['Anchor', 'Comparable']:
        update_checkbox_anchor_comp(grid, clicked_bond, col_name)
        update_anchor_comparable(grid, spread_history, spread_fig, lookback)

In [17]:
def create_spread_curve_chart(bond_grid_data, spread_col, title):
    """
    Create spread vs maturity curve (today's snapshot).
    X-axis: Maturity in years (to nearest 0.25)
    Y-axis: Spread
    """
    df = bond_grid_data.copy()
    df = df.dropna(subset=[spread_col, 'Mty Yrs'])
    
    # Replace any Inf values
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=[spread_col])
    
    df = df.sort_values('Mty Yrs')
    
    # Round maturity to nearest 0.25 years (3 months)
    df['Mty Rounded'] = (df['Mty Yrs'] * 4).round() / 4
    
    fig = go.FigureWidget()
    
    if len(df) > 0:
        fig.add_trace(
            go.Scatter(
                x=df['Mty Rounded'].tolist(),
                y=df[spread_col].tolist(),
                mode='lines+markers',
                text=df['Name'].tolist(),
                hovertemplate='<b>%{text}</b><br>Maturity: %{x:.2f} yrs<br>Spread: %{y:.1f} bps<extra></extra>',
                line=dict(width=2),
                marker=dict(size=8)
            )
        )
    
    fig.update_layout(
        template='plotly_dark',
        title=dict(text=title, font=dict(size=14)),
        xaxis_title='Maturity (Years)',
        yaxis_title='Spread (bps)',
        height=280,
        margin=dict(t=40, r=10, b=40, l=50),
        showlegend=False
    )
    
    return fig

In [18]:
def create_heatmap_figure(diff_matrix, zscore_matrix, title):
    """
    Create heatmap showing spread differences and z-scores.
    Shows longer maturity - shorter maturity.
    """
    # Create labels with shortened names
    labels = [n.replace('SAGB ', '').replace('SOAF ', '') for n in diff_matrix.index]
    
    # Replace NaN with None for proper JSON serialization
    zscore_values = zscore_matrix.values.copy()
    zscore_values = np.where(np.isnan(zscore_values), None, zscore_values)
    
    # Create hover text with diff and z-score
    hover_text = []
    for i, row_name in enumerate(diff_matrix.index):
        row_hover = []
        for j, col_name in enumerate(diff_matrix.columns):
            diff_val = diff_matrix.iloc[i, j]
            z_val = zscore_matrix.iloc[i, j]
            if pd.isna(diff_val):
                row_hover.append('')
            else:
                z_str = f'{z_val:.2f}' if not pd.isna(z_val) else 'N/A'
                row_hover.append(f'{row_name} - {col_name}<br>Diff: {diff_val:.1f}<br>Z: {z_str}')
        hover_text.append(row_hover)
    
    fig = go.FigureWidget()
    
    fig.add_trace(
        go.Heatmap(
            z=zscore_values.tolist(),
            x=labels,
            y=labels,
            colorscale='RdBu_r',
            zmid=0,
            zmin=-3,
            zmax=3,
            text=hover_text,
            hovertemplate='%{text}<extra></extra>',
            colorbar=dict(title='Z-Score', len=0.8)
        )
    )
    
    fig.update_layout(
        template='plotly_dark',
        title=dict(text=title, font=dict(size=14)),
        xaxis_title='Short Maturity',
        yaxis_title='Long Maturity',
        height=400,
        margin=dict(t=40, r=10, b=80, l=100),
        xaxis=dict(tickangle=45, tickfont=dict(size=9)),
        yaxis=dict(tickfont=dict(size=9)),
    )
    
    return fig

In [19]:
# Containers for dynamic content
grid_box = widgets.Box(layout={'width': '100%', 'margin': '10px 0'})
charts_box = widgets.VBox(layout={'width': '100%'})

In [20]:
def run(event=None):
    """
    Main workflow.
    """
    spinner.layout.visibility = 'visible'
    grid_box.children = []
    charts_box.children = []
    exception.value = ''
   
    try:
        ticker_name = ticker.value
        spread_type = spread_chart_type.value
        hist_start = start_date.value
        lookback = zscore_lookback.value
    
        # Get bond data
        bond_grid_data = get_bond_grid_data(ticker_name)
        print(f"Retrieved {len(bond_grid_data)} bonds")
    
        # Get historical data
        spread_history = generate_spread_history(ticker_name, spread_type, bond_grid_data, hist_start)
        if not spread_history.empty:
            print(f"Retrieved {spread_type} history: {len(spread_history)} dates")
        else:
            print(f"Warning: No {spread_type} history retrieved")
        
        lqa_history = generate_lqa_history(ticker_name, bond_grid_data, hist_start)
        if not lqa_history.empty:
            print(f"Retrieved LQA history: {len(lqa_history)} dates")
        else:
            print("Warning: No LQA history retrieved")
        
        # Fetch both ASW and Z spread histories for heatmaps
        asw_history = spread_history if spread_type == 'ASW' else generate_spread_history(ticker_name, 'ASW', bond_grid_data, hist_start)
        z_history = spread_history if spread_type == 'Z Spd' else generate_spread_history(ticker_name, 'Z Spd', bond_grid_data, hist_start)
        
        # Create grid
        grid = create_grid(bond_grid_data)
        
        # Create spread history chart with subplots
        spread_fig = make_subplots(
            rows=3, cols=1,
            vertical_spacing=0.12,
            row_heights=[0.5, 0.25, 0.25],
            subplot_titles=(f'{spread_type} Spread History', 'Difference', 'Z-Score'),
            shared_xaxes=True
        )
        
        # Add empty traces for diff and z-score with distinct colors (not in bond color palette)
        spread_fig.add_trace(go.Scatter(name="Diff", line=dict(width=1.5, color='#888888'), showlegend=False), row=2, col=1)
        spread_fig.add_trace(go.Scatter(name="Z", line=dict(width=1.5, color='#888888'), showlegend=False), row=3, col=1)
        
        spread_fig.update_layout(
            template='plotly_dark',
            height=450,
            margin=dict(t=30, r=150, b=10, l=50),
            legend=dict(
                orientation="v",
                yanchor="top", y=1.0,
                xanchor="left", x=1.02,
                font=dict(size=10),
                bgcolor='rgba(0,0,0,0.5)'
            ),
            hovermode="x unified",
            showlegend=True
        )
        spread_fig.update_yaxes(title_text='bps', row=1, col=1)
        spread_fig.update_yaxes(title_text='Diff', zeroline=True, zerolinecolor='gray', row=2, col=1)
        spread_fig.update_yaxes(title_text='Z', zeroline=True, zerolinecolor='gray', row=3, col=1)
        spread_fig = go.FigureWidget(spread_fig)
        
        # Create LQA chart
        lqa_fig = go.FigureWidget()
        lqa_fig.update_layout(
            template='plotly_dark',
            title=dict(text='LQA Score History', font=dict(size=14)),
            height=350,
            margin=dict(t=40, r=150, b=10, l=50),
            legend=dict(
                orientation="v",
                yanchor="top", y=1.0,
                xanchor="left", x=1.02,
                font=dict(size=10),
                bgcolor='rgba(0,0,0,0.5)'
            ),
            hovermode="x unified",
            yaxis_title='LQA'
        )
        
        # Create spread curves (today's snapshot)
        asw_curve = create_spread_curve_chart(bond_grid_data, 'ASW', 'ASW Spread vs Maturity (Today)')
        z_curve = create_spread_curve_chart(bond_grid_data, 'Z Spd', 'Z-Spread vs Maturity (Today)')
        
        # Create heatmaps using the fetched histories
        asw_diff, asw_zscore = create_spread_heatmap_data(bond_grid_data, asw_history, 'ASW', lookback)
        z_diff, z_zscore = create_spread_heatmap_data(bond_grid_data, z_history, 'Z Spd', lookback)
        
        asw_heatmap = create_heatmap_figure(asw_diff, asw_zscore, 'ASW Spread Diff Z-Score (Long - Short)')
        z_heatmap = create_heatmap_figure(z_diff, z_zscore, 'Z-Spread Diff Z-Score (Long - Short)')
        
        # Wire up grid clicks
        grid.on_cell_click(partial(on_grid_click, grid, spread_history, lqa_history, spread_fig, lqa_fig, lookback))
        
        # Assemble layout
        grid_box.children = [grid]
        
        charts_box.children = [
            widgets.HTML('<hr style="margin: 15px 0;">'),
            widgets.HBox([spread_fig, lqa_fig], layout={'width': '100%'}),
            widgets.HTML('<hr style="margin: 15px 0;">'),
            widgets.HBox([asw_curve, z_curve], layout={'width': '100%'}),
            widgets.HTML('<hr style="margin: 15px 0;">'),
            widgets.HBox([asw_heatmap, z_heatmap], layout={'width': '100%'}),
        ]

    except Exception as e:
        import traceback
        exception.value = f"<span style='color:red;'>Error: {e}</span>"
        traceback.print_exc()

    finally:
        spinner.layout.visibility = 'hidden'

In [21]:
# Register button - click Go to load data
go_button.on_click(run)
# Uncomment below to auto-run on notebook load:
# run()

In [22]:
# Display
widgets.VBox([control_box, grid_box, charts_box], layout={'width': '100%'})